# A-7 (배치판) — 적응형 공격: 앙상블 S(x) 전체 타깃 + c=0 sanity check


In [ ]:
import os, sys, subprocess, pickle, io as _io
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

SEARCH_ROOTS=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
def _find(name, ftype='f', maxdepth=8):
    res=[]
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        try:
            out=subprocess.run(['find',root,'-maxdepth',str(maxdepth),'-type',ftype,'-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406];   IMGNET_STD=[0.229,0.224,0.225]
def make_preprocess(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1); std=torch.tensor(s).view(1,3,1,1)
    return lambda x:(x/255.0-mean.to(x.device))/std.to(x.device)
def load_backbone(ds):
    if ds=='CIFAR-10':
        ck=(_find('resnet50_cifar10_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    elif ds=='CIFAR-100':
        ck=(_find('resnet50_cifar100_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,100)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=100
    elif ds=='SVHN':
        ck=(_find('resnet50_svhn_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,10)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=10
    elif ds in ('TinyImageNet','Tiny ImageNet'):
        ck=(_find('resnet50_tinyimagenet_finetuned.pt') or [None])[0]
        m=models.resnet50(weights=None); m.fc=nn.Linear(2048,200)
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict']); nc=200
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2); nc=1000
    return m.to(device).eval(), nc
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'cifar100' in pl: out.setdefault('CIFAR-100',p)
        elif 'svhn' in pl: out.setdefault('SVHN',p)
        elif 'tiny' in pl: out.setdefault('TinyImageNet',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet_eps8',p)
    return out
def gb(x,sigma):
    k=int(2*np.ceil(3*sigma)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=sigma)
def to224(img,mode='bicubic'):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224 or img.shape[-2]!=224:
        img=F.interpolate(img,size=(224,224),mode=mode,align_corners=False)
    return img.clamp(0,255)
def jpeg(img224,q):
    arr=img224.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    buf=_io.BytesIO(); _Image.fromarray(arr).save(buf,format='JPEG',quality=int(q)); buf.seek(0)
    return torch.from_numpy(np.array(_Image.open(buf).convert('RGB'))).float().permute(2,0,1).unsqueeze(0).to(img224.device)
def jpeg_batch(x,q=75):
    return torch.cat([jpeg(x[i:i+1],q) for i in range(x.shape[0])],0)
def median3(x):
    xx=F.pad(x,(1,1,1,1),mode='reflect'); p=xx.unfold(2,3,1).unfold(3,3,1)
    return p.contiguous().view(*p.shape[:4],9).median(dim=-1).values
def load_mixed(pkl_path, n_clean=500):
    with open(pkl_path,'rb') as f: mixed=pickle.load(f)
    clean=[im for (im,lb,atk) in mixed if atk=='clean']
    adv  =[(im,atk) for (im,lb,atk) in mixed if atk!='clean']
    rng=np.random.RandomState(SEED); idx=np.arange(len(clean)); rng.shuffle(idx)
    return [clean[i] for i in idx[:n_clean]], adv
print('header ready; device=',device)

In [ ]:
DS='ImageNet'   # 'CIFAR-10' 또는 'ImageNet' (ImageNet은 OOM시 N_ADAPT 낮추기)
N_ADAPT=32; STEPS=80; LR=0.02
MX=find_mixed(); mkey='CIFAR-10' if DS=='CIFAR-10' else 'ImageNet_eps8'
backbone,nc=load_backbone(DS); pp=make_preprocess(DS)
clean,_=load_mixed(MX[mkey],n_clean=400)
# 정확히 분류되는 이미지만
imgs=[]
for im in clean:
    x=to224(im).to(device)
    with torch.no_grad():
        if backbone(pp(x)).argmax(1).item()==0 or True:  # keep all; label below
            imgs.append(x)
    if len(imgs)>=N_ADAPT+160: break
X=torch.cat(imgs,0)
with torch.no_grad(): Y=backbone(pp(X)).argmax(1)
Xatk=X[:N_ADAPT]; Yatk=Y[:N_ADAPT]; Xcal=X[N_ADAPT:N_ADAPT+128]
print('attack batch:',Xatk.shape,'calib:',Xcal.shape)

In [ ]:
# 미분가능 배치 특징 + 캘리브레이션
def hfe_b(x): return (x-gb(x,0.5)).abs().flatten(1).mean(1)/255.0      # [N]
def gl_b(x,p0=None):
    if p0 is None: p0=F.softmax(backbone(pp(x)),1)
    p1=F.softmax(backbone(pp(gb(x,1.0))),1)
    return (p0-p1).abs().sum(1)
def predl1_b(x,p0=None):
    if p0 is None: p0=F.softmax(backbone(pp(x)),1)
    sq=median3(jpeg_batch(x,75)).clamp(0,255); sq=x+(sq-x).detach()    # BPDA
    p1=F.softmax(backbone(pp(sq)),1)
    return (p0-p1).abs().sum(1)
with torch.no_grad():
    H=hfe_b(Xcal); p0=F.softmax(backbone(pp(Xcal)),1); G=gl_b(Xcal,p0); P=predl1_b(Xcal,p0)
MU={'h':H.mean().item(),'g':G.mean().item(),'p':P.mean().item()}
SD={'h':H.std().item()+1e-8,'g':G.std().item()+1e-8,'p':P.std().item()+1e-8}
def S_batch(x):
    p0=F.softmax(backbone(pp(x)),1)
    zh=(hfe_b(x)-MU['h'])/SD['h']; zg=(gl_b(x,p0)-MU['g'])/SD['g']; zp=(predl1_b(x,p0)-MU['p'])/SD['p']
    return (zh+zg+zp)/3.0,(zh,zg,zp)
print('calibrated',MU)

In [ ]:
def adaptive_batch(X0,Yv,c,lam,steps=STEPS,lr=LR):
    Xn=(X0/255.0).clamp(1e-4,1-1e-4); w=torch.atanh(Xn*2-1).clone().requires_grad_(True)
    opt=torch.optim.Adam([w],lr=lr)
    for t in range(steps):
        xadv=(torch.tanh(w)+1)/2*255.0
        logits=backbone(pp(xadv))
        zt=logits.gather(1,Yv.view(-1,1)).squeeze(1)
        oth=logits.clone(); oth.scatter_(1,Yv.view(-1,1),-1e9)
        margin=torch.clamp(zt-oth.max(1).values,min=0.0)
        S,_=S_batch(xadv)
        l2=((xadv-X0)/255.0).pow(2).flatten(1).sum(1)
        loss=(l2 + c*margin + lam*(S**2)).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    xadv=((torch.tanh(w)+1)/2*255.0).detach()
    with torch.no_grad():
        succ=(backbone(pp(xadv)).argmax(1)!=Yv).float().mean().item()
        l2=((xadv-X0)/255.0).pow(2).flatten(1).sum(1).sqrt().mean().item()
        S,(zh,zg,zp)=S_batch(xadv)
    return xadv,succ,l2,S.abs(),(zh.mean().item(),zg.mean().item(),zp.mean().item())
with torch.no_grad(): Sclean,_=S_batch(Xatk); negS=Sclean.abs().cpu().numpy()
def run(c,lam,tag):
    _,asr,l2,Spos,z=adaptive_batch(Xatk,Yatk,c,lam)
    pos=Spos.cpu().numpy()
    a=roc_auc_score(np.r_[np.zeros(len(negS)),np.ones(len(pos))],np.r_[negS,pos])
    print(f"  {tag:<26} ASR={asr:.2f} L2={l2:.3f} detAUC={a:.4f} z(HF/GL/PL)={z[0]:+.2f}/{z[1]:+.2f}/{z[2]:+.2f}")
    return dict(tag=tag,asr=asr,l2=l2,auc=a,z=list(z))
print(f"[{DS}] 적응형 (배치 N={N_ADAPT}, steps={STEPS})")
RES=[run(1.0,0.0,'Standard C&W (λ=0)')]
for lam in [0.5,1.0,5.0,10.0]: RES.append(run(1.0,lam,f'Adaptive C&W (λ={lam})'))
RES.append(run(0.0,5.0,'c=0 Evasion-only (λ=5)'))
Path('./adaptive_v4_results').mkdir(exist_ok=True)
pickle.dump(RES,open('./adaptive_v4_results/adaptive_v4.pkl','wb'))
print("\n판정: c=0 detAUC≈0.5 → gradient 정상. λ↑에도 detAUC 유지 시 z(HF/GL/PL)로 회피 특징 분해.")